##### Using Python-Tracked SQL

In [ ]:
df = spark.read.format("csv").option("header","true").load("Files/product/products.csv")
display(df)

In [ ]:
%%sql

select * from product

In [ ]:
from delta.tables import DeltaTable
from datetime import datetime
from pyspark.sql.functions import lit
import uuid

nof_records = df.count()

operation_id = str(uuid.uuid4())

source_with_id = df.withColumn("operation_id", lit(operation_id))

source_with_id.createOrReplaceTempView("source_table")




In [ ]:
# Execute the SQL MERGE query in Python and capture the metrics
metrics_df = spark.sql("""
MERGE INTO product AS target
USING source_table AS source
ON target.productid = source.productid
WHEN MATCHED AND source.is_modified == 1 THEN
  UPDATE SET
    target.name = source.name,
    target.category = source.category,
    target.price = source.price,
    target.operation_id = source.operation_id
WHEN NOT MATCHED THEN
  INSERT (productid, name, category, Subcategory, brand, description, price, color, operation_id)
  VALUES (source.productid, source.name, source.category, source.sub_category, source.brand
    , source.description, source.price, source.color, source.operation_id)
""")

# Display the metrics DataFrame
metrics_df.show()



In [ ]:
if nof_records > 0:

    if metrics_df.count() > 0:        

        num_inserted = metrics_df.collect()[0]["num_inserted_rows"]
        num_updated = metrics_df.collect()[0]["num_updated_rows"]

        # Print metrics
        print("Records updated")
        print("Number of rows inserted: ", num_inserted)
        print("Number of rows updated: ", num_updated)
    else:
        print("No operations performed")
        print("Number of rows inserted: 0")
        print("Number of rows updated: 0")
else:
    print("No records to update")